In [1]:
import atomic_units as au
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
from quantum_toolkit import potentials as pots
import quantum_toolkit as quat
import scipy.sparse as sparse
import numpy as np
import time
import threading

display(HTML("""
<style>
.output_scroll {overflow: visible !important; max-height: none !important;}
.potparam-row > .widget-label { min-width: 90px; }
.potparam-row > .widget-label, .potparam-row > .widget-inline-hbox { margin-right: 8px; }
</style>
"""))

# --- Globális paraméterek ---
hb=1.0
m0=8.8
k0=2.3
e0=3.4
unit_sys = au.AtomicUnitSystem(xh=hb, xe=e0, xm=m0, xk=k0)

# --- Unit system widgetek deklarációja (KORÁBBRA MOZGATVA) ---
hbar_widget = widgets.FloatText(description="hbar:", value=unit_sys.hb)
m0_widget = widgets.FloatText(description="m0:", value=unit_sys.me)
kappa0_widget = widgets.FloatText(description="kappa0:", value=unit_sys.k0)
e0_widget = widgets.FloatText(description="e0:", value=unit_sys.e0)

uxgrid_num = 4*1024
uxgrid_dx = 0.025
uxgrid_width = uxgrid_num * uxgrid_dx
uxgrid = np.arange(-uxgrid_width/2, uxgrid_width/2, uxgrid_dx)

# --- Itt legyen a dx_widget definíció ---
dx_widget = widgets.FloatText(description="Lépésköz:", value=uxgrid_dx, min=1e-10)

# Paraméter widgetek
wallwidth_widget = widgets.FloatText(value=3.0, description="wallwidth [nm]")
wallheight_widget = widgets.FloatText(value=1.0, description="wallheight [eV]")
wallrise_widget = widgets.FloatText(value=1.0, description="wallrise [nm]")
wellwidth_widget = widgets.FloatText(value=10.0, description="wellwidth [nm]")
welldepth_widget = widgets.FloatText(value=1.0, description="welldepth [eV]")
wellfall_widget = widgets.FloatText(value=1.0, description="wellfall [nm]")

# Új widgetek a szuperrácshoz
num_cells_widget = widgets.IntText(value=1, min=1, description="Number of cells:", style={'description_width': 'initial'})
cell_spacing_widget = widgets.FloatText(value=0.0, description="Cell spacing [nm]:", style={'description_width': 'initial'})

def get_modelpot():
    wallwidth = wallwidth_widget.value * unit_sys.convert_length_from('nm')
    wallheight = wallheight_widget.value * unit_sys.convert_energy_from('eV')
    wallrise = wallrise_widget.value
    wellwidth = wellwidth_widget.value * unit_sys.convert_length_from('nm')
    welldepth = welldepth_widget.value * unit_sys.convert_energy_from('eV')
    wellfall = wellfall_widget.value
    num_cells = num_cells_widget.value
    cell_spacing = cell_spacing_widget.value * unit_sys.convert_length_from('nm')

    # Egy cella szélessége (2*fal + gödör + spacing)
    cell_length_nm = 2*wallwidth_widget.value + wellwidth_widget.value + cell_spacing_widget.value
    cell_length_unit = cell_length_nm * unit_sys.convert_length_from('nm')

    pot = np.zeros_like(uxgrid)
    for i in range(num_cells):
        offset = (i - (num_cells-1)/2) * cell_length_unit
        cell_pot = pots.SmoothStackPotential(
            uxgrid - offset,
            wallwidth=wallwidth,
            wallheight=wallheight,
            welldepth=welldepth,
            wellwidth=wellwidth,
            wallrise=wallrise,
            wellfall=wellfall
        )(0)
        pot += cell_pot #pot = np.maximum(pot, cell_pot)
    return lambda t=0: pot

# --- Potential tab ---
plot_unit_toggle = widgets.ToggleButtons(
    options=[('unit_sys', False), ('nm/eV', True)],
    value=True,
    description='Plot units:',
    style={'description_width': 'initial'}
)
plot_margin_widget = widgets.FloatText(value=10.0, layout=widgets.Layout(width='70px'))
plot_margin_label = widgets.Label("nm")
pot_output = widgets.Output()
pot_output.layout = widgets.Layout(overflow='visible', max_height='none')

def plot_potential(change=None):
    with pot_output:
        clear_output(wait=True)
        modelpot = get_modelpot()
        plot_margin_nm = plot_margin_widget.value
        total_width_nm = num_cells_widget.value * (2*wallwidth_widget.value + wellwidth_widget.value + cell_spacing_widget.value)
        x_min_nm = -0.5 * total_width_nm - plot_margin_nm
        x_max_nm = 0.5 * total_width_nm + plot_margin_nm

        if plot_unit_toggle.value:
            x_plot = uxgrid * unit_sys.length_unit.to('nm').magnitude
            y = modelpot(0) * unit_sys.energy_unit.to('eV').magnitude
            x_min = x_min_nm
            x_max = x_max_nm
            xlabel = "x [nm]"
            ylabel = "Potenciál [eV]"
        else:
            x_plot = uxgrid
            x_min = (x_min_nm / unit_sys.length_unit.to('nm').magnitude)
            x_max = (x_max_nm / unit_sys.length_unit.to('nm').magnitude)
            xlabel = "x [unit_sys]"
            ylabel = "Potenciál [unit_sys]"
            y = modelpot(0)

        plt.figure(figsize=(7, 3))
        plt.plot(x_plot, y)
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.grid()
        plt.xlim([x_min, x_max])
        plt.tight_layout()
        plt.show()

for w in [
    wallwidth_widget, wallheight_widget, wallrise_widget,
    wellwidth_widget, welldepth_widget, wellfall_widget,
    plot_unit_toggle, plot_margin_widget,
    num_cells_widget, cell_spacing_widget
]:
    w.observe(plot_potential, names='value')
plot_potential()

potential_tab = widgets.VBox([
    widgets.Label("SmoothStackPotential parameters"),
    plot_unit_toggle,
    widgets.HBox([widgets.Label("Plot margin:"), plot_margin_widget, plot_margin_label]),
    num_cells_widget,
    cell_spacing_widget,
    wallwidth_widget, wallheight_widget, wallrise_widget,
    wellwidth_widget, welldepth_widget, wellfall_widget,
    pot_output
])

# --- Eigenstate tab ---
eigen_output = widgets.Output()

num_eigen_widget = widgets.IntText(
    value=6, min=1, description="Number of eigenvalues to compute:",
    style={'description_width': 'initial'}
)
num_plot_widget = widgets.IntText(
    value=6, min=1, description="Show on plot:",
    style={'description_width': 'initial'}
)

eigen_unit_toggle = widgets.ToggleButtons(
    options=[('unit_sys', False), ('nm/eV', True)],
    value=True,
    description='Plot units:',
    style={'description_width': 'initial'}
)

# Tároljuk az utolsó eredményt
_last_eigenvalues = None
_last_eigenvectors = None
_last_x_plot = None
_last_y_pot = None
_last_eigenvalues_plot = None
_last_xlabel = None
_last_ylabel = None

def compute_eigenstates(_=None):
    global _last_eigenvalues, _last_eigenvectors, _last_x_plot, _last_y_pot, _last_eigenvalues_plot, _last_xlabel, _last_ylabel
    with eigen_output:
        clear_output(wait=True)
        print("Eigenstate calculation started...")
        import time
        start_time = time.time()
        wallwidth = wallwidth_widget.value * unit_sys.convert_length_from('nm')
        wallheight = wallheight_widget.value * unit_sys.convert_energy_from('eV')
        welldepth = welldepth_widget.value * unit_sys.convert_energy_from('eV')
        wellwidth = wellwidth_widget.value * unit_sys.convert_length_from('nm')
        wallrise = wallrise_widget.value
        wellfall = wellfall_widget.value
        modelpot = get_modelpot()
        zeropot = pots.ZeroPotential(uxgrid)
        ham_static = quat.HamiltonOperator(
            uxgrid,
            hbar=unit_sys.hb,
            me=unit_sys.me,
            charge=unit_sys.e0,
            scalarpot=modelpot,
            vectorpot=zeropot
        )
        num_eigen = max(1, int(num_eigen_widget.value))
        num_plot = max(1, int(num_plot_widget.value))
        print(f"Diagonalization in progress for {num_eigen} eigenvalues, please wait...")
        eigenvalues, eigenvectors = sparse.linalg.eigsh(ham_static(0), num_eigen, which='SR')
        elapsed = time.time() - start_time

        # Mértékegység választás
        if eigen_unit_toggle.value:
            x_plot = uxgrid * unit_sys.length_unit.to('nm').magnitude
            y_pot = modelpot(0) * unit_sys.energy_unit.to('eV').magnitude
            eigenvalues_plot = eigenvalues * unit_sys.energy_unit.to('eV').magnitude
            xlabel = "x [nm]"
            ylabel = "Energy [eV]"
        else:
            x_plot = uxgrid
            y_pot = modelpot(0)
            eigenvalues_plot = eigenvalues
            xlabel = "x [unit_sys]"
            ylabel = "Energy [unit_sys]"

        # Eredmények eltárolása
        _last_eigenvalues = eigenvalues
        _last_eigenvectors = eigenvectors
        _last_x_plot = x_plot
        _last_y_pot = y_pot
        _last_eigenvalues_plot = eigenvalues_plot
        _last_xlabel = xlabel
        _last_ylabel = ylabel

        print(f"Calculation finished in {elapsed:.1f} seconds.")
        print(f"Eigenvalues:", eigenvalues_plot[:num_plot])
        plot_eigenstates(num_plot)

def plot_eigenstates(num_plot=None):
    with eigen_output:
        clear_output(wait=True)
        if _last_eigenvalues is None:
            print("No eigenstate data. Please compute eigenstates first.")
            return
        if num_plot is None:
            num_plot = max(1, int(num_plot_widget.value))
        x_plot = _last_x_plot
        y_pot = _last_y_pot
        eigenvalues_plot = _last_eigenvalues_plot
        eigenvectors = _last_eigenvectors
        xlabel = _last_xlabel
        ylabel = _last_ylabel
        num_eigen = len(_last_eigenvalues)

        print(f"Eigenvalues:", eigenvalues_plot[:num_plot])

        plt.figure(figsize=(8, 4))
        plt.plot(x_plot, y_pot, label="Potential")
        for eval, evec in zip(eigenvalues_plot[:num_plot], np.transpose(eigenvectors)[:num_plot]):
            plt.plot(x_plot, eval * np.ones_like(x_plot), "--", label=f"E={eval:.2e}")
        plt.xlabel(xlabel)
        plt.xlim([x_plot[0], x_plot[-1]])
        plt.legend(loc='upper right')
        plt.grid()
        plt.title("Eigenstates and Potential")
        plt.show()

        plt.figure()
        plt.scatter(np.arange(num_eigen)[:num_plot], eigenvalues_plot[:num_plot], marker='o', color='red')
        plt.xlabel("State index")
        plt.ylabel(ylabel)
        plt.title("Eigenvalues")
        plt.grid()
        plt.show()

eigen_button = widgets.Button(description="Compute eigenstates", button_style='primary')
eigen_button.on_click(compute_eigenstates)
num_plot_widget.observe(lambda change: plot_eigenstates(), names='value')
eigen_unit_toggle.observe(lambda change: plot_eigenstates(), names='value')

# Foton hullámhossz és energia mezők
photon_wavelength_widget = widgets.FloatText(
    value=800.0, description="Photon wavelength [nm]:", style={'description_width': 'initial'}
)
photon_energy_widget = widgets.Label(value="Photon energy: --- eV")

def update_photon_energy(change=None):
    # E = hc/λ, ahol h = 4.135667696e-15 eV·s, c = 299792458 m/s, λ nm-ben
    h = 4.135667696e-15  # eV·s
    c = 299792458  # m/s
    try:
        wavelength_nm = photon_wavelength_widget.value
        wavelength_m = wavelength_nm * 1e-9
        energy_eV = h * c / wavelength_m
        photon_energy_widget.value = f"Photon energy: {energy_eV:.3f} eV"
    except Exception:
        photon_energy_widget.value = "Photon energy: --- eV"

photon_wavelength_widget.observe(update_photon_energy, names='value')
update_photon_energy()

eigen_tab = widgets.VBox([
    widgets.Label("Eigenstate calculation"),
    eigen_unit_toggle,
    num_eigen_widget,
    num_plot_widget,
    photon_wavelength_widget,
    photon_energy_widget,
    widgets.HBox([eigen_button]),
    eigen_output
])

# --- Dispersion tab ---
# dispersion = None  # Ezt töröljük, helyette globális változóként kezeljük

# Dispersion objektum automatikus létrehozása a kezdeti paraméterekkel
dispersion = None
def update_dispersion(*args):
    global dispersion
    import quantum_toolkit as quat
    try:
        dispersion = quat.CosineDispersion(
            dx_widget.value,
            hbar=unit_sys.hb,
            mass=unit_sys.me
        )
    except Exception as e:
        print("Hiba a dispersion létrehozásakor:", e)
update_dispersion()  # Induláskor egyszer lefut

# Frissítés, ha dx_widget vagy unit_sys paraméterek változnak
dx_widget.observe(lambda change: update_dispersion(), names='value')
hbar_widget.observe(lambda change: update_dispersion(), names='value')
m0_widget.observe(lambda change: update_dispersion(), names='value')

dispersion_output = widgets.Output()

def create_dispersion(_=None):
    with dispersion_output:
        clear_output(wait=True)
        update_dispersion()
        print("Dispersion objektum létrehozva/frissítve:")
        print("dx =", dx_widget.value)
        print("hbar =", unit_sys.hb)
        print("mass =", unit_sys.me)

dispersion_button = widgets.Button(description="Create dispersion", button_style='primary')
dispersion_button.on_click(create_dispersion)
dispersion_tab = widgets.VBox([
    widgets.Label("Dispersion objektum létrehozása/frissítése az aktuális beállításokkal"),
    dispersion_button,
    dispersion_output
])

# --- Egyéb tabok (unit, grid, scatter) ---
unit_tab = widgets.VBox([
    widgets.Label("Atomic Unit system"),
    hbar_widget,
    m0_widget,
    kappa0_widget,
    e0_widget
])
points_widget = widgets.IntText(description="Pontok száma:", value=uxgrid_num, min=1)
dx_widget = widgets.FloatText(description="Lépésköz:", value=uxgrid_dx, min=1e-10)
width_widget = widgets.Label(value="Teljes hossz: ---")
def update_width(*args):
    try:
        n = points_widget.value
        dx = dx_widget.value
        width = n * dx * unit_sys.length_unit.to('nm')
        width_widget.value = f"Teljes hossz: {width:.4f}"
    except Exception:
        width_widget.value = "Teljes hossz: ---"
points_widget.observe(update_width, names='value')
dx_widget.observe(update_width, names='value')
update_width()
grid_tab = widgets.VBox([
    widgets.Label("Grid setup"),
    points_widget,
    dx_widget,
    width_widget
])

# --- Scatter tab számítás és gomb összekötése ---
scatter_output = widgets.Output()
quasiparticle_selector = widgets.Dropdown(
    options=[],
    description="Quasiparticle:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)
show_wave_output = widgets.Output()
progress_bar = widgets.FloatProgress(value=0.0, min=0.0, max=1.0, description='Progress:', bar_style='info', layout=widgets.Layout(width='350px'))

# Állapotkezelő a számítás megszakításához
scatter_calc_state = {'running': False, 'thread': None}

def compute_scatter_thread():
    with scatter_output:
        clear_output(wait=True)
        t_vals = []
        r_vals = []
        quasiparticles_pk = []
        quasiparticles_mk = []
        qp_labels = []

        # Paraméterek a dispersion fülről
        energy_min = disp_energy_min.value
        energy_max = disp_energy_max.value
        num_points = disp_number_of_points.value

        energy_min_in_unit_sys = energy_min * unit_sys.convert_energy_from('eV')
        energy_max_in_unit_sys = energy_max * unit_sys.convert_energy_from('eV')

        # K-grid vagy energy-grid alapján
        if disp_grid_mode.value == 'unified in k':
            k_min = dispersion.wavenumber(energy_min_in_unit_sys)
            k_max = dispersion.wavenumber(energy_max_in_unit_sys)
            k_vals = np.linspace(k_min, k_max, num_points)
            e_vals = [dispersion.energy(k) for k in k_vals]
        else:
            e_vals = np.linspace(energy_min_in_unit_sys, energy_max_in_unit_sys, num_points)
            k_vals = [dispersion.wavenumber(e) for e in e_vals]

        # Potenciál és Hamilton-operátor
        modelpot = get_modelpot()
        zeropot = pots.ZeroPotential(uxgrid)
        ham_static = quat.HamiltonOperator(
            uxgrid,
            hbar=unit_sys.hb,
            me=unit_sys.me,
            charge=unit_sys.e0,
            scalarpot=modelpot,
            vectorpot=zeropot
        )

        for idx, (ein, kin) in enumerate(zip(e_vals, k_vals)):
            if not scatter_calc_state['running']:
                progress_bar.value = 0.0
                progress_bar.bar_style = 'warning'
                return
            qp_pk, qp_mk = quat.quasiparticles_at_given_energy(ein, kin, ham_static)
            quasiparticles_pk.append(qp_pk)
            quasiparticles_mk.append(qp_mk)
            t_vals.append(qp_pk['transmission'])
            r_vals.append(qp_pk['reflection'])
            qp_labels.append(f"E={ein:.3e} k={kin:.3e} T={qp_pk['transmission']:.2f}")
            progress_bar.value = (idx + 1) / num_points

            # --- Frissítsd a dropdown-t ÉS az adatokat minden lépésben ---
            quasiparticle_selector.options = [(label, i) for i, label in enumerate(qp_labels)]
            quasiparticle_selector._quasiparticles = list(quasiparticles_pk)
            quasiparticle_selector._modelpot = modelpot
            if quasiparticle_selector.value is None and qp_labels:
                quasiparticle_selector.value = 0

        # Ábrázolás
        plt.figure()
        plt.plot(k_vals, t_vals, "-o", label="T")
        plt.plot(k_vals, r_vals, "-o", label="R")
        plt.xlabel("k [arb. u.]")
        plt.legend()
        plt.grid()
        plt.title("Transmission/Reflection vs k")
        plt.show()

        plt.figure()
        plt.plot(e_vals, t_vals, "-o", label="T")
        plt.plot(e_vals, r_vals, "-o", label="R")
        plt.xlabel("electron energy [arb. u.]")
        plt.legend()
        plt.grid()
        plt.title("Transmission/Reflection vs energy")
        plt.show()

        progress_bar.value = 1.0
        progress_bar.bar_style = 'success'

# def compute_scatter_thread():
#     with scatter_output:
#         clear_output(wait=True)
#         t_vals = []
#         r_vals = []
#         quasiparticles_pk = []
#         quasiparticles_mk = []
#         qp_labels = []

#         # Paraméterek a dispersion fülről
#         energy_min = disp_energy_min.value
#         energy_max = disp_energy_max.value
#         num_points = disp_number_of_points.value

#         energy_min_in_unit_sys = energy_min * unit_sys.convert_energy_from('eV')
#         energy_max_in_unit_sys = energy_max * unit_sys.convert_energy_from('eV')

#         # K-grid vagy energy-grid alapján
#         if disp_grid_mode.value == 'unified in k':
#             k_min = dispersion.wavenumber(energy_min_in_unit_sys)
#             k_max = dispersion.wavenumber(energy_max_in_unit_sys)
#             k_vals = np.linspace(k_min, k_max, num_points)
#             e_vals = [dispersion.energy(k) for k in k_vals]
#         else:
#             e_vals = np.linspace(energy_min_in_unit_sys, energy_max_in_unit_sys, num_points)
#             k_vals = [dispersion.wavenumber(e) for e in e_vals]

#         # Potenciál és Hamilton-operátor
#         modelpot = get_modelpot()
#         zeropot = pots.ZeroPotential(uxgrid)
#         ham_static = quat.HamiltonOperator(
#             uxgrid,
#             hbar=unit_sys.hb,
#             me=unit_sys.me,
#             charge=unit_sys.e0,
#             scalarpot=modelpot,
#             vectorpot=zeropot
#         )

#         for idx, (ein, kin) in enumerate(zip(e_vals, k_vals)):
#             if not scatter_calc_state['running']:
#                 progress_bar.value = 0.0
#                 progress_bar.bar_style = 'warning'
#                 return
#             qp_pk, qp_mk = quat.quasiparticles_at_given_energy(ein, kin, ham_static)
#             quasiparticles_pk.append(qp_pk)
#             quasiparticles_mk.append(qp_mk)
#             t_vals.append(qp_pk['transmission'])
#             r_vals.append(qp_pk['reflection'])
#             qp_labels.append(f"E={ein:.3e} k={kin:.3e} T={qp_pk['transmission']:.2f}")
#             progress_bar.value = (idx + 1) / num_points

#         # Frissítsd a dropdown-t
#         quasiparticle_selector.options = [(label, i) for i, label in enumerate(qp_labels)]
#         quasiparticle_selector.value = 0 if qp_labels else None

#         # Ábrázolás
#         plt.figure()
#         plt.plot(k_vals, t_vals, "-o", label="T")
#         plt.plot(k_vals, r_vals, "-o", label="R")
#         plt.xlabel("k [arb. u.]")
#         plt.legend()
#         plt.grid()
#         plt.title("Transmission/Reflection vs k")
#         plt.show()

#         plt.figure()
#         plt.plot(e_vals, t_vals, "-o", label="T")
#         plt.plot(e_vals, r_vals, "-o", label="R")
#         plt.xlabel("electron energy [arb. u.]")
#         plt.legend()
#         plt.grid()
#         plt.title("Transmission/Reflection vs energy")
#         plt.show()

#         # Tárold a kvázipartikulumokat a selector callback-hez
#         quasiparticle_selector._quasiparticles = quasiparticles_pk
#         quasiparticle_selector._modelpot = modelpot
#         progress_bar.value = 1.0
#         progress_bar.bar_style = 'success'

def compute_scatter(_=None):
    if scatter_calc_state['running']:
        # Stop gomb megnyomva
        scatter_calc_state['running'] = False
        scatter_button.description = "Compute"
        scatter_button.button_style = 'primary'
        progress_bar.bar_style = 'danger'
        return
    # Start számítás
    scatter_calc_state['running'] = True
    scatter_button.description = "Stop"
    scatter_button.button_style = 'danger'
    progress_bar.value = 0.0
    progress_bar.bar_style = 'info'
    thread = threading.Thread(target=compute_scatter_thread)
    scatter_calc_state['thread'] = thread
    thread.start()

def plot_selected_quasiparticle(change=None):
    idx = quasiparticle_selector.value
    quasiparticles = getattr(quasiparticle_selector, "_quasiparticles", None)
    modelpot = getattr(quasiparticle_selector, "_modelpot", None)
    if quasiparticles is None or modelpot is None or idx is None:
        return
    qp = quasiparticles[idx]
    with show_wave_output:
        clear_output(wait=True)
        plt.figure(figsize=(8, 4))
        plt.plot(uxgrid, modelpot(0), label="Potential")
        plt.plot(uxgrid, np.real(qp["state"]["value"]), label="Re")
        plt.plot(uxgrid, np.imag(qp["state"]["value"]), label="Im")
        plt.plot(uxgrid, np.abs(qp["state"]["value"]), label="Abs")
        plt.xlabel("x [arb. u.]")
        plt.legend()
        plt.grid()
        plt.title("Selected quasiparticle wavefunction")
        plt.show()

quasiparticle_selector.observe(plot_selected_quasiparticle, names='value')

scatter_button = widgets.Button(description="Compute", button_style='primary')
scatter_button.on_click(compute_scatter)
scatter_tab = widgets.VBox([
    widgets.Label("Scatter"),
    scatter_button,
    progress_bar,
    quasiparticle_selector,
    scatter_output,
    show_wave_output
])

# --- Dispersion tab interaktív ábra ---
disp_grid_mode = widgets.Dropdown(
    options=['unified in energy', 'unified in k'],
    value='unified in k',
    description='Grid type:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)
disp_energy_min = widgets.FloatText(
    value=0.02,
    description='min:',
    min=0.0,  # Csak pozitív érték engedélyezett
    style={'description_width': '60px'},
    layout=widgets.Layout(width='180px')
)
disp_energy_max = widgets.FloatText(
    value=1.2,
    description='max:',
    min=0.0,  # Csak pozitív érték engedélyezett
    style={'description_width': '60px'},
    layout=widgets.Layout(width='180px')
)
disp_number_of_points = widgets.IntText(
    value=50,
    min=2,
    description='Num. of pts.:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='180px')
)

def update_dispersion_tab_plot(grid_mode, energy_min, energy_max, num):
    import matplotlib.pyplot as plt
    e_min = energy_min
    e_max = energy_max
    if dispersion is None:
        print("A dispersion objektum nincs inicializálva. Kérlek, hozd létre először a 'Create dispersion' gombbal!")
        return
    if grid_mode == 'unified in k':
        k_min = dispersion.wavenumber(e_min)
        k_max = dispersion.wavenumber(e_max)
        kvals, dk = np.linspace(k_min, k_max, num, retstep=True)
        evals = dispersion.energy(kvals)
        plt.plot(kvals, evals, 'o-', label='k-grid')
        plt.xlabel("k [arb. u.]")
        plt.ylabel("energy [arb. u.]")
        plt.title("Equidistant in k")
    else:
        evals = np.linspace(e_min, e_max, num)
        kvals = dispersion.wavenumber(evals)
        plt.plot(kvals, evals, 'o-', label='energy-grid')
        plt.xlabel("k [arb. u.]")
        plt.ylabel("energy [arb. u.]")
        plt.title("Equidistant in energy")
    plt.grid()
    plt.legend()
    plt.show()

disp_wg_plot = widgets.interactive_output(
    update_dispersion_tab_plot,
    {
        'grid_mode': disp_grid_mode,
        'energy_min': disp_energy_min,
        'energy_max': disp_energy_max,
        'num': disp_number_of_points
    }
)

dispersion_tab = widgets.VBox([
    widgets.Label("Dispersion grid setup"),
    disp_grid_mode,
    disp_energy_min,
    disp_energy_max,
    disp_number_of_points,
    disp_wg_plot
])

# Új Scatter tab (csak ábrázolás, nem számítás)
scatter_plot_mode = widgets.ToggleButtons(
    options=[('k', 'k'), ('energy', 'energy')],
    value='k',
    description='X axis:',
    style={'description_width': 'initial'}
)
scatter_plot_output = widgets.Output()

def plot_scatter_transmission_reflection(change=None):
    with scatter_plot_output:
        clear_output(wait=True)
        # Csak akkor engedélyezett, ha a compute tabban már van quasiparticle lista
        quasiparticles = getattr(quasiparticle_selector, "_quasiparticles", None)
        if not quasiparticles or len(quasiparticles) == 0:
            print("Előbb futtasd le a Compute tabban a számítást!")
            return

        # A compute tabban eltárolt k, e, t, r értékekből ábrázolunk
        # Feltételezzük, hogy compute_scatter_thread-ben ezek az értékek a selectorhoz vannak rendelve
        qp_labels = getattr(quasiparticle_selector, "options", [])
        # A compute tabban minden label tartalmazza az E, k, T értékeket
        # Újraépítjük a listákat
        k_vals = []
        e_vals = []
        t_vals = []
        r_vals = []
        for i, (label, idx) in enumerate(qp_labels):
            # Példa label: "E=1.634e-01 k=1.696e+00 T=0.00"
            try:
                parts = label.split()
                e_val = float(parts[0].split('=')[1])
                k_val = float(parts[1].split('=')[1])
                t_val = float(parts[2].split('=')[1])
                # Reflectiont a compute tabban a quasiparticle dictből tudjuk
                r_val = quasiparticles[idx].get('reflection', 0)
                e_vals.append(e_val)
                k_vals.append(k_val)
                t_vals.append(t_val)
                r_vals.append(r_val)
            except Exception:
                continue

        if scatter_plot_mode.value == 'k':
            plt.plot(k_vals, t_vals, "-o", label="T")
            plt.plot(k_vals, r_vals, "-o", label="R")
            plt.xlabel("k [arb. u.]")
        else:
            plt.plot(e_vals, t_vals, "-o", label="T")
            plt.plot(e_vals, r_vals, "-o", label="R")
            plt.xlabel("electron energy [arb. u.]")
        plt.legend()
        plt.grid()
        plt.ylabel("Coefficient")
        plt.title("Transmission/Reflection")
        plt.show()

scatter_plot_mode.observe(plot_scatter_transmission_reflection, names='value')
# Első kirajzolás
plot_scatter_transmission_reflection()

scatter_tab2 = widgets.VBox([
    widgets.Label("Scatter plot"),
    scatter_plot_mode,
    scatter_plot_output
])

# --- Tabs összerakása egy sorban ---
tabs = widgets.Tab(children=[
    unit_tab, grid_tab, potential_tab, eigen_tab,
    dispersion_tab, scatter_tab, scatter_tab2
])
tab_titles = [
    'Unit system', 'Grid', 'Potential', 'Eigenstates',
    'Dispersion', 'Compute', 'Scatter'
]
for i, title in enumerate(tab_titles):
    tabs.set_title(i, title)

display(tabs)